# Magistra — część badawcza (100%)

**Temat:** Detekcja ataków inżynierii społecznej i prób manipulacji semantycznej w komunikacji cyfrowej z wykorzystaniem architektury Transformer

## Cel eksperymentu
Porównać **klasyczny baseline** (TF-IDF + LinearSVC) oraz **3 modele Transformer**:
1. `bert-base-uncased`
2. `roberta-base`
3. `distilbert-base-uncased`

na **5 publicznych korpusach** (bez zbierania własnych danych).

## Metryki
- główna przy imbalance: **F1 macro**
- dodatkowo: accuracy, precision, recall, F1 (klasa pozytywna)

## Środowisko
1. **Środowisko wykonawcze → Zmień typ → GPU T4 → Zapisz**
2. Runtime → **Uruchom wszystko**
3. Czas: EDA+SVM ~5–15 min; 3 Transformery na 5 zbiorach ~**2–4 h** na T4 (nie zamykaj karty)

## Zakres (potwierdzony z promotorem)
- liczby baz **wystarczy**
- nie tylko BERT — **2–3 Transformery**


## 0) GPU check + zależności
Sprawdź, że widać **Tesla T4** / CUDA = True.


In [ ]:
!nvidia-smi
%pip install -q scikit-learn pandas numpy transformers accelerate huggingface_hub
import torch
print("cuda:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "BRAK GPU — ustaw T4 i zrestartuj runtime")


## 1) Kod projektu + pobranie baz
Komórka zapisuje pipeline w `/content/se-transformer` i ściąga 5 korpusów.
Ma retry przy HTTP 429 (Hugging Face).


In [ ]:
from pathlib import Path
import json, time, shutil, urllib.request, urllib.error

ROOT = Path("/content/se-transformer")
SRC = ROOT / "src"
SRC.mkdir(parents=True, exist_ok=True)
(ROOT / "outputs").mkdir(exist_ok=True)
(ROOT / "models").mkdir(exist_ok=True)
for sub in ["seconvo", "mentalmanip", "safepersuasion", "scam_phone", "phishing"]:
    (ROOT / "data" / sub).mkdir(parents=True, exist_ok=True)

FILES = json.loads("{\"paths.py\": \"from pathlib import Path\\n\\nROOT = Path(__file__).resolve().parents[1]\\nDATA = ROOT / \\\"data\\\"\\nOUTPUTS = ROOT / \\\"outputs\\\"\\nMODELS = ROOT / \\\"models\\\"\\n\\nSECONVO_DIR = DATA / \\\"seconvo\\\"\\nMENTAL_DIR = DATA / \\\"mentalmanip\\\"\\nSAFE_DIR = DATA / \\\"safepersuasion\\\"\\nSCAM_DIR = DATA / \\\"scam_phone\\\"\\nPHISH_DIR = DATA / \\\"phishing\\\"\\n\\nSECONVO_TRAIN = SECONVO_DIR / \\\"annotated_train.json\\\"\\nSECONVO_TEST = SECONVO_DIR / \\\"annotated_test.json\\\"\\nMENTAL_CSV = MENTAL_DIR / \\\"mentalmanip_maj.csv\\\"\\nSAFE_CSV = SAFE_DIR / \\\"SafePersuasion.csv\\\"\\nSCAM_TRAIN = SCAM_DIR / \\\"agent_conversation_train.csv\\\"\\nSCAM_TEST = SCAM_DIR / \\\"agent_conversation_test.csv\\\"\\nPHISH_JSON = PHISH_DIR / \\\"texts.json\\\"\\n\\nCORE_TASKS = [\\\"seconvo\\\", \\\"mentalmanip\\\"]\\nEXTRA_TASKS = [\\\"safepersuasion\\\", \\\"scam_phone\\\", \\\"phishing_text\\\"]\\nALL_TASKS = CORE_TASKS + EXTRA_TASKS\\n\\n# Three Transformers for the thesis research chapter (promotor: 2\u20133 models).\\nTRANSFORMER_MODELS = [\\n    \\\"bert-base-uncased\\\",\\n    \\\"roberta-base\\\",\\n    \\\"distilbert-base-uncased\\\",\\n]\\n\", \"download.py\": \"\\\"\\\"\\\"Download all thesis corpora (core + extra). Retries on 429.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport shutil\\nimport time\\nimport urllib.error\\nimport urllib.request\\nfrom pathlib import Path\\n\\nfrom src.paths import (\\n    MENTAL_CSV,\\n    MENTAL_DIR,\\n    PHISH_DIR,\\n    PHISH_JSON,\\n    SAFE_CSV,\\n    SAFE_DIR,\\n    SCAM_DIR,\\n    SCAM_TEST,\\n    SCAM_TRAIN,\\n    SECONVO_DIR,\\n    SECONVO_TEST,\\n    SECONVO_TRAIN,\\n)\\n\\nFILES: list[tuple[str, Path]] = [\\n    (\\n        \\\"https://zenodo.org/records/12170260/files/annotated_train.json?download=1\\\",\\n        SECONVO_TRAIN,\\n    ),\\n    (\\n        \\\"https://zenodo.org/records/12170260/files/annotated_test.json?download=1\\\",\\n        SECONVO_TEST,\\n    ),\\n    (\\n        \\\"https://huggingface.co/datasets/audreyeleven/MentalManip/resolve/main/mentalmanip_maj.csv?download=true\\\",\\n        MENTAL_CSV,\\n    ),\\n    (\\n        \\\"https://raw.githubusercontent.com/haeinkong/SafePersuasion/main/dataset/SafePersuasion.csv\\\",\\n        SAFE_CSV,\\n    ),\\n    (\\n        \\\"https://huggingface.co/datasets/BothBosu/multi-agent-scam-conversation/resolve/main/agent_conversation_train.csv?download=true\\\",\\n        SCAM_TRAIN,\\n    ),\\n    (\\n        \\\"https://huggingface.co/datasets/BothBosu/multi-agent-scam-conversation/resolve/main/agent_conversation_test.csv?download=true\\\",\\n        SCAM_TEST,\\n    ),\\n    (\\n        \\\"https://huggingface.co/datasets/ealvaradob/phishing-dataset/resolve/main/texts.json?download=true\\\",\\n        PHISH_JSON,\\n    ),\\n]\\n\\n\\ndef _fetch_urllib(url: str, dest: Path, attempts: int = 6) -> None:\\n    tmp = dest.with_suffix(dest.suffix + \\\".part\\\")\\n    last_err: Exception | None = None\\n    for i in range(attempts):\\n        try:\\n            req = urllib.request.Request(url, headers={\\\"User-Agent\\\": \\\"thesis-se-transformer/1.0\\\"})\\n            with urllib.request.urlopen(req, timeout=120) as resp, tmp.open(\\\"wb\\\") as out:\\n                shutil.copyfileobj(resp, out)\\n            shutil.move(tmp, dest)\\n            return\\n        except urllib.error.HTTPError as exc:\\n            last_err = exc\\n            if tmp.exists():\\n                tmp.unlink(missing_ok=True)\\n            wait = 15 * (i + 1)\\n            if exc.code in (429, 503):\\n                print(f\\\"  HTTP {exc.code}, wait {wait}s ({i + 1}/{attempts})\\\")\\n                time.sleep(wait)\\n                continue\\n            raise\\n        except Exception as exc:  # noqa: BLE001\\n            last_err = exc\\n            if tmp.exists():\\n                tmp.unlink(missing_ok=True)\\n            wait = 10 * (i + 1)\\n            print(f\\\"  error {exc}, wait {wait}s ({i + 1}/{attempts})\\\")\\n            time.sleep(wait)\\n    raise RuntimeError(f\\\"failed to download {dest.name}: {last_err}\\\")\\n\\n\\ndef _fetch_hf_hub(repo_file: str, dest: Path) -> bool:\\n    \\\"\\\"\\\"repo_file like 'ealvaradob/phishing-dataset@texts.json'.\\\"\\\"\\\"\\n    try:\\n        from huggingface_hub import hf_hub_download\\n    except ImportError:\\n        return False\\n    repo_id, filename = repo_file.split(\\\"@\\\", 1)\\n    path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type=\\\"dataset\\\")\\n    shutil.copy(path, dest)\\n    return True\\n\\n\\ndef _fetch(url: str, dest: Path) -> None:\\n    dest.parent.mkdir(parents=True, exist_ok=True)\\n    if dest.exists() and dest.stat().st_size > 1000:\\n        print(f\\\"skip {dest.name} ({dest.stat().st_size} bytes)\\\")\\n        return\\n    print(f\\\"download {dest.name}\\\")\\n    # Prefer HF hub for large HF files (better rate limits).\\n    if \\\"ealvaradob/phishing-dataset\\\" in url and \\\"texts.json\\\" in url:\\n        if _fetch_hf_hub(\\\"ealvaradob/phishing-dataset@texts.json\\\", dest):\\n            print(f\\\"  -> {dest.stat().st_size} bytes (hf_hub)\\\")\\n            return\\n    _fetch_urllib(url, dest)\\n    print(f\\\"  -> {dest.stat().st_size} bytes\\\")\\n\\n\\ndef main() -> None:\\n    for folder in (SECONVO_DIR, MENTAL_DIR, SAFE_DIR, SCAM_DIR, PHISH_DIR):\\n        folder.mkdir(parents=True, exist_ok=True)\\n    for url, dest in FILES:\\n        _fetch(url, dest)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"data_load.py\": \"\\\"\\\"\\\"Load labeled texts for core + extra thesis tasks.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport csv\\nimport json\\nfrom dataclasses import dataclass\\n\\nfrom sklearn.model_selection import train_test_split\\n\\nfrom src.paths import (\\n    ALL_TASKS,\\n    MENTAL_CSV,\\n    PHISH_JSON,\\n    SAFE_CSV,\\n    SCAM_TEST,\\n    SCAM_TRAIN,\\n    SECONVO_TEST,\\n    SECONVO_TRAIN,\\n)\\n\\nSEED = 42\\n\\n\\n@dataclass(frozen=True)\\nclass Split:\\n    task: str\\n    texts_train: list[str]\\n    y_train: list[int]\\n    texts_test: list[str]\\n    y_test: list[int]\\n    ids_train: list[str]\\n    ids_test: list[str]\\n\\n\\ndef _dialogue_to_text(turns: list[dict]) -> str:\\n    lines: list[str] = []\\n    for turn in turns:\\n        name = str(turn.get(\\\"Name\\\", \\\"Speaker\\\")).strip()\\n        msg = str(turn.get(\\\"Message\\\", \\\"\\\")).strip()\\n        if msg:\\n            lines.append(f\\\"{name}: {msg}\\\")\\n    return \\\"\\\\n\\\".join(lines)\\n\\n\\ndef load_seconvo_all() -> tuple[list[str], list[int], list[str]]:\\n    texts: list[str] = []\\n    labels: list[int] = []\\n    ids: list[str] = []\\n    for path in (SECONVO_TRAIN, SECONVO_TEST):\\n        raw = json.loads(path.read_text(encoding=\\\"utf-8\\\"))\\n        for item in raw[\\\"Conversations\\\"]:\\n            gt = item[\\\"GroundTruth\\\"]\\n            cid = str(gt.get(\\\"ConversationID\\\", len(ids)))\\n            texts.append(_dialogue_to_text(item[\\\"Conversation\\\"]))\\n            labels.append(1 if gt[\\\"IsMalicious\\\"] else 0)\\n            ids.append(cid)\\n    return texts, labels, ids\\n\\n\\ndef load_mentalmanip_all() -> tuple[list[str], list[int], list[str]]:\\n    texts: list[str] = []\\n    labels: list[int] = []\\n    ids: list[str] = []\\n    with MENTAL_CSV.open(encoding=\\\"utf-8\\\", newline=\\\"\\\") as handle:\\n        reader = csv.DictReader(handle)\\n        for row in reader:\\n            texts.append(row[\\\"dialogue\\\"].strip())\\n            labels.append(int(row[\\\"manipulative\\\"]))\\n            ids.append(str(row[\\\"id\\\"]))\\n    return texts, labels, ids\\n\\n\\ndef load_safepersuasion_all() -> tuple[list[str], list[int], list[str]]:\\n    \\\"\\\"\\\"Manipulation=1, Rational Persuasion=0 (semantic manipulation in online comments).\\\"\\\"\\\"\\n    texts: list[str] = []\\n    labels: list[int] = []\\n    ids: list[str] = []\\n    with SAFE_CSV.open(encoding=\\\"utf-8\\\", newline=\\\"\\\") as handle:\\n        reader = csv.DictReader(handle)\\n        for i, row in enumerate(reader):\\n            label_name = row[\\\"first_label\\\"].strip()\\n            if label_name == \\\"Manipulation\\\":\\n                y = 1\\n            elif label_name == \\\"Rational Persuasion\\\":\\n                y = 0\\n            else:\\n                continue\\n            text = row[\\\"text\\\"].strip()\\n            if not text:\\n                continue\\n            texts.append(text)\\n            labels.append(y)\\n            ids.append(f\\\"safe_{i}\\\")\\n    return texts, labels, ids\\n\\n\\ndef _normalize_scam_dialogue(text: str) -> str:\\n    # Strip role tags that are identical across classes (reduce template leakage).\\n    return (\\n        text.replace(\\\"Innocent:\\\", \\\"Receiver:\\\")\\n        .replace(\\\"Suspect:\\\", \\\"Caller:\\\")\\n        .strip()\\n    )\\n\\n\\ndef load_scam_phone_split() -> Split:\\n    \\\"\\\"\\\"Official train/test from BothBosu multi-agent scam conversations.\\\"\\\"\\\"\\n\\n    def _read(path, prefix: str) -> tuple[list[str], list[int], list[str]]:\\n        texts: list[str] = []\\n        labels: list[int] = []\\n        ids: list[str] = []\\n        with path.open(encoding=\\\"utf-8\\\", newline=\\\"\\\") as handle:\\n            for i, row in enumerate(csv.DictReader(handle)):\\n                texts.append(_normalize_scam_dialogue(row[\\\"dialogue\\\"]))\\n                labels.append(int(row[\\\"labels\\\"]))\\n                ids.append(f\\\"{prefix}_{i}\\\")\\n        return texts, labels, ids\\n\\n    x_tr, y_tr, id_tr = _read(SCAM_TRAIN, \\\"tr\\\")\\n    x_te, y_te, id_te = _read(SCAM_TEST, \\\"te\\\")\\n    return Split(\\\"scam_phone\\\", x_tr, y_tr, x_te, y_te, id_tr, id_te)\\n\\n\\ndef load_phishing_text_all() -> tuple[list[str], list[int], list[str]]:\\n    \\\"\\\"\\\"Email/SMS-style phishing vs benign (ealvaradob texts subset).\\\"\\\"\\\"\\n    raw = json.loads(PHISH_JSON.read_text(encoding=\\\"utf-8\\\"))\\n    texts: list[str] = []\\n    labels: list[int] = []\\n    ids: list[str] = []\\n    seen: set[str] = set()\\n    for i, row in enumerate(raw):\\n        text = str(row.get(\\\"text\\\", \\\"\\\")).strip()\\n        if not text:\\n            continue\\n        # Drop exact duplicates before split (avoid train/test leakage).\\n        if text in seen:\\n            continue\\n        seen.add(text)\\n        texts.append(text)\\n        labels.append(int(row[\\\"label\\\"]))\\n        ids.append(f\\\"phish_{i}\\\")\\n    return texts, labels, ids\\n\\n\\ndef stratified_split(\\n    task: str,\\n    texts: list[str],\\n    labels: list[int],\\n    ids: list[str],\\n    test_size: float = 0.2,\\n    limit: int | None = None,\\n) -> Split:\\n    if limit is not None:\\n        # stratified subsample before split so smoke stays class-balanced\\n        if len(texts) > limit:\\n            texts, _, labels, _, ids, _ = train_test_split(\\n                texts,\\n                labels,\\n                ids,\\n                train_size=limit,\\n                random_state=SEED,\\n                stratify=labels if len(set(labels)) > 1 else None,\\n            )\\n    x_tr, x_te, y_tr, y_te, id_tr, id_te = train_test_split(\\n        texts,\\n        labels,\\n        ids,\\n        test_size=test_size,\\n        random_state=SEED,\\n        stratify=labels if len(set(labels)) > 1 else None,\\n    )\\n    return Split(task, x_tr, y_tr, x_te, y_te, id_tr, id_te)\\n\\n\\ndef load_task(task: str, limit: int | None = None) -> Split:\\n    if task not in ALL_TASKS:\\n        raise ValueError(f\\\"unknown task: {task}; expected one of {ALL_TASKS}\\\")\\n    if task == \\\"scam_phone\\\":\\n        split = load_scam_phone_split()\\n        if limit is None:\\n            return split\\n        # smoke: shrink train/test proportionally\\n        n_tr = max(8, int(limit * 0.8))\\n        n_te = max(4, limit - n_tr)\\n        return Split(\\n            split.task,\\n            split.texts_train[:n_tr],\\n            split.y_train[:n_tr],\\n            split.texts_test[:n_te],\\n            split.y_test[:n_te],\\n            split.ids_train[:n_tr],\\n            split.ids_test[:n_te],\\n        )\\n    if task == \\\"seconvo\\\":\\n        texts, labels, ids = load_seconvo_all()\\n    elif task == \\\"mentalmanip\\\":\\n        texts, labels, ids = load_mentalmanip_all()\\n    elif task == \\\"safepersuasion\\\":\\n        texts, labels, ids = load_safepersuasion_all()\\n    elif task == \\\"phishing_text\\\":\\n        texts, labels, ids = load_phishing_text_all()\\n    else:\\n        raise ValueError(f\\\"unknown task: {task}\\\")\\n    return stratified_split(task, texts, labels, ids, limit=limit)\\n\", \"metrics.py\": \"from __future__ import annotations\\n\\nfrom collections import Counter\\n\\nfrom sklearn.metrics import (\\n    accuracy_score,\\n    classification_report,\\n    confusion_matrix,\\n    f1_score,\\n    precision_score,\\n    recall_score,\\n)\\n\\n\\ndef score(y_true: list[int], y_pred: list[int]) -> dict[str, float | list | str]:\\n    return {\\n        \\\"accuracy\\\": float(accuracy_score(y_true, y_pred)),\\n        \\\"precision\\\": float(precision_score(y_true, y_pred, zero_division=0)),\\n        \\\"recall\\\": float(recall_score(y_true, y_pred, zero_division=0)),\\n        \\\"f1\\\": float(f1_score(y_true, y_pred, zero_division=0)),\\n        \\\"f1_macro\\\": float(f1_score(y_true, y_pred, average=\\\"macro\\\", zero_division=0)),\\n        \\\"confusion_matrix\\\": confusion_matrix(y_true, y_pred).tolist(),\\n        \\\"report\\\": classification_report(y_true, y_pred, digits=4, zero_division=0),\\n    }\\n\\n\\ndef majority_baseline(y_true: list[int]) -> dict[str, float | int]:\\n    maj = Counter(y_true).most_common(1)[0][0]\\n    pred = [maj] * len(y_true)\\n    return {\\n        \\\"majority_class\\\": int(maj),\\n        \\\"accuracy\\\": float(accuracy_score(y_true, pred)),\\n        \\\"f1\\\": float(f1_score(y_true, pred, zero_division=0)),\\n        \\\"f1_macro\\\": float(f1_score(y_true, pred, average=\\\"macro\\\", zero_division=0)),\\n    }\\n\", \"train_svm.py\": \"from __future__ import annotations\\n\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.svm import LinearSVC\\n\\nfrom src.data_load import Split\\nfrom src.metrics import score\\n\\n\\ndef train_eval_svm(split: Split, max_features: int = 50_000) -> dict:\\n    pipe = Pipeline(\\n        [\\n            (\\n                \\\"tfidf\\\",\\n                TfidfVectorizer(\\n                    lowercase=True,\\n                    ngram_range=(1, 2),\\n                    min_df=2,\\n                    max_features=max_features,\\n                ),\\n            ),\\n            (\\\"clf\\\", LinearSVC(class_weight=\\\"balanced\\\", random_state=42, max_iter=4000)),\\n        ]\\n    )\\n    pipe.fit(split.texts_train, split.y_train)\\n    pred = pipe.predict(split.texts_test).tolist()\\n    metrics = score(split.y_test, pred)\\n    return {\\n        \\\"task\\\": split.task,\\n        \\\"model\\\": \\\"tfidf_linearsvc\\\",\\n        \\\"n_train\\\": len(split.y_train),\\n        \\\"n_test\\\": len(split.y_test),\\n        \\\"metrics\\\": metrics,\\n    }\\n\", \"train_bert.py\": \"from __future__ import annotations\\n\\nimport numpy as np\\nimport torch\\nfrom torch.utils.data import Dataset\\nfrom transformers import (\\n    AutoModelForSequenceClassification,\\n    AutoTokenizer,\\n    Trainer,\\n    TrainingArguments,\\n)\\n\\nfrom src.data_load import Split\\nfrom src.metrics import score\\nfrom src.paths import MODELS\\n\\n\\nclass TextClsDataset(Dataset):\\n    def __init__(self, encodings: dict, labels: list[int]) -> None:\\n        self.encodings = encodings\\n        self.labels = labels\\n\\n    def __len__(self) -> int:\\n        return len(self.labels)\\n\\n    def __getitem__(self, idx: int) -> dict:\\n        item = {key: val[idx] for key, val in self.encodings.items()}\\n        item[\\\"labels\\\"] = torch.tensor(self.labels[idx], dtype=torch.long)\\n        return item\\n\\n\\ndef train_eval_bert(\\n    split: Split,\\n    model_name: str = \\\"bert-base-uncased\\\",\\n    epochs: float = 3,\\n    batch_size: int = 8,\\n    max_length: int = 256,\\n    lr: float = 2e-5,\\n) -> dict:\\n    tokenizer = AutoTokenizer.from_pretrained(model_name)\\n    train_enc = tokenizer(\\n        split.texts_train,\\n        truncation=True,\\n        padding=True,\\n        max_length=max_length,\\n    )\\n    test_enc = tokenizer(\\n        split.texts_test,\\n        truncation=True,\\n        padding=True,\\n        max_length=max_length,\\n    )\\n    train_ds = TextClsDataset(train_enc, split.y_train)\\n    test_ds = TextClsDataset(test_enc, split.y_test)\\n\\n    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)\\n    out_dir = MODELS / f\\\"{split.task}_{model_name.replace('/', '_')}\\\"\\n    common = {\\n        \\\"output_dir\\\": str(out_dir),\\n        \\\"num_train_epochs\\\": epochs,\\n        \\\"per_device_train_batch_size\\\": batch_size,\\n        \\\"per_device_eval_batch_size\\\": batch_size,\\n        \\\"learning_rate\\\": lr,\\n        \\\"save_strategy\\\": \\\"no\\\",\\n        \\\"logging_steps\\\": 20,\\n        \\\"report_to\\\": [],\\n        \\\"seed\\\": 42,\\n        \\\"fp16\\\": torch.cuda.is_available(),\\n        \\\"dataloader_pin_memory\\\": False,\\n    }\\n    try:\\n        args = TrainingArguments(**common, eval_strategy=\\\"epoch\\\")\\n    except TypeError:\\n        args = TrainingArguments(**common, evaluation_strategy=\\\"epoch\\\")\\n\\n    def compute_metrics(eval_pred: tuple) -> dict[str, float]:\\n        logits, labels = eval_pred\\n        preds = np.argmax(logits, axis=-1)\\n        m = score(labels.tolist(), preds.tolist())\\n        return {\\n            \\\"accuracy\\\": m[\\\"accuracy\\\"],\\n            \\\"precision\\\": m[\\\"precision\\\"],\\n            \\\"recall\\\": m[\\\"recall\\\"],\\n            \\\"f1\\\": m[\\\"f1\\\"],\\n            \\\"f1_macro\\\": m[\\\"f1_macro\\\"],\\n        }\\n\\n    trainer = Trainer(\\n        model=model,\\n        args=args,\\n        train_dataset=train_ds,\\n        eval_dataset=test_ds,\\n        compute_metrics=compute_metrics,\\n    )\\n    trainer.train()\\n    pred = np.argmax(trainer.predict(test_ds).predictions, axis=-1).tolist()\\n    metrics = score(split.y_test, pred)\\n    return {\\n        \\\"task\\\": split.task,\\n        \\\"model\\\": model_name,\\n        \\\"n_train\\\": len(split.y_train),\\n        \\\"n_test\\\": len(split.y_test),\\n        \\\"device\\\": \\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\",\\n        \\\"metrics\\\": metrics,\\n    }\\n\", \"run_all.py\": \"\\\"\\\"\\\"Run majority + SVM + one or more Transformers. Writes Markdown/JSON tables.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom datetime import datetime, timezone\\n\\nfrom src.data_load import load_task\\nfrom src.metrics import majority_baseline\\nfrom src.paths import ALL_TASKS, CORE_TASKS, EXTRA_TASKS, OUTPUTS, TRANSFORMER_MODELS\\nfrom src.train_svm import train_eval_svm\\n\\n\\ndef _md_row(row: dict) -> str:\\n    m = row[\\\"metrics\\\"]\\n    return (\\n        f\\\"| {row['task']} | {row['model']} | {row['n_train']} | {row['n_test']} | \\\"\\n        f\\\"{m['accuracy']:.4f} | {m['precision']:.4f} | {m['recall']:.4f} | \\\"\\n        f\\\"{m['f1']:.4f} | {m['f1_macro']:.4f} |\\\"\\n    )\\n\\n\\ndef write_tables(rows: list[dict], tag: str) -> None:\\n    OUTPUTS.mkdir(parents=True, exist_ok=True)\\n    stamp = datetime.now(timezone.utc).strftime(\\\"%Y%m%dT%H%M%SZ\\\")\\n    json_path = OUTPUTS / f\\\"results_{tag}_{stamp}.json\\\"\\n    md_path = OUTPUTS / f\\\"results_{tag}_{stamp}.md\\\"\\n    latest_md = OUTPUTS / \\\"results_latest.md\\\"\\n    json_path.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding=\\\"utf-8\\\")\\n    header = (\\n        \\\"| task | model | n_train | n_test | accuracy | precision | recall | f1 | f1_macro |\\\\n\\\"\\n        \\\"|---|---|---:|---:|---:|---:|---:|---:|---:|\\\\n\\\"\\n    )\\n    body = \\\"\\\\n\\\".join(_md_row(r) for r in rows)\\n    reports = \\\"\\\\n\\\\n\\\".join(\\n        f\\\"### {r['task']} / {r['model']}\\\\n\\\\n```\\\\n{r['metrics']['report']}\\\\n```\\\"\\n        for r in rows\\n    )\\n    md = f\\\"# Results `{tag}`\\\\n\\\\n{header}{body}\\\\n\\\\n{reports}\\\\n\\\"\\n    md_path.write_text(md, encoding=\\\"utf-8\\\")\\n    latest_md.write_text(md, encoding=\\\"utf-8\\\")\\n    print(md)\\n    print(f\\\"wrote {json_path}\\\")\\n    print(f\\\"wrote {md_path}\\\")\\n\\n\\ndef _resolve_tasks(name: str) -> list[str]:\\n    if name == \\\"core\\\":\\n        return list(CORE_TASKS)\\n    if name == \\\"extra\\\":\\n        return list(EXTRA_TASKS)\\n    if name == \\\"all\\\":\\n        return list(ALL_TASKS)\\n    if name == \\\"both\\\":\\n        return list(CORE_TASKS)\\n    if name in ALL_TASKS:\\n        return [name]\\n    raise ValueError(f\\\"unknown task selector: {name}\\\")\\n\\n\\ndef _resolve_models(raw: str | None, skip_transformers: bool, smoke: bool) -> list[str]:\\n    if skip_transformers:\\n        return []\\n    if smoke:\\n        return [\\\"distilbert-base-uncased\\\"]\\n    if not raw or raw.strip() == \\\"all\\\":\\n        return list(TRANSFORMER_MODELS)\\n    return [m.strip() for m in raw.split(\\\",\\\") if m.strip()]\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\n        \\\"--task\\\",\\n        default=\\\"all\\\",\\n        help=\\\"seconvo|mentalmanip|safepersuasion|scam_phone|phishing_text|core|extra|all\\\",\\n    )\\n    parser.add_argument(\\\"--limit\\\", type=int, default=None, help=\\\"subset for smoke tests\\\")\\n    parser.add_argument(\\\"--skip-bert\\\", action=\\\"store_true\\\", help=\\\"alias: skip transformers\\\")\\n    parser.add_argument(\\\"--skip-transformers\\\", action=\\\"store_true\\\")\\n    parser.add_argument(\\n        \\\"--models\\\",\\n        default=\\\"all\\\",\\n        help=\\\"comma list or 'all' \u2192 bert-base-uncased,roberta-base,distilbert-base-uncased\\\",\\n    )\\n    parser.add_argument(\\\"--bert-model\\\", default=None, help=\\\"deprecated: single model (use --models)\\\")\\n    parser.add_argument(\\\"--epochs\\\", type=float, default=3)\\n    parser.add_argument(\\\"--batch-size\\\", type=int, default=8)\\n    parser.add_argument(\\\"--max-length\\\", type=int, default=512)\\n    parser.add_argument(\\\"--smoke\\\", action=\\\"store_true\\\")\\n    args = parser.parse_args()\\n\\n    skip_tr = args.skip_bert or args.skip_transformers\\n    if args.smoke:\\n        args.limit = args.limit or 64\\n        args.epochs = 1\\n        args.batch_size = 4\\n        args.max_length = 128\\n\\n    models = _resolve_models(args.bert_model or args.models, skip_tr, args.smoke)\\n    tasks = _resolve_tasks(args.task)\\n    rows: list[dict] = []\\n\\n    for task in tasks:\\n        split = load_task(task, limit=args.limit)\\n        print(f\\\"\\\\n=== {task} train={len(split.y_train)} test={len(split.y_test)} ===\\\")\\n        maj = majority_baseline(split.y_test)\\n        print(\\n            f\\\"majority baseline: class={maj['majority_class']} \\\"\\n            f\\\"acc={maj['accuracy']:.4f} f1={maj['f1']:.4f} f1_macro={maj['f1_macro']:.4f}\\\"\\n        )\\n        rows.append(\\n            {\\n                \\\"task\\\": task,\\n                \\\"model\\\": \\\"majority_baseline\\\",\\n                \\\"n_train\\\": len(split.y_train),\\n                \\\"n_test\\\": len(split.y_test),\\n                \\\"metrics\\\": {\\n                    \\\"accuracy\\\": maj[\\\"accuracy\\\"],\\n                    \\\"precision\\\": 0.0,\\n                    \\\"recall\\\": 0.0,\\n                    \\\"f1\\\": maj[\\\"f1\\\"],\\n                    \\\"f1_macro\\\": maj[\\\"f1_macro\\\"],\\n                    \\\"confusion_matrix\\\": [],\\n                    \\\"report\\\": f\\\"majority_class={maj['majority_class']}\\\",\\n                },\\n            }\\n        )\\n        rows.append(train_eval_svm(split))\\n        if not models:\\n            continue\\n        from src.train_bert import train_eval_bert\\n\\n        for model_name in models:\\n            print(f\\\"--- transformer: {model_name} ---\\\")\\n            rows.append(\\n                train_eval_bert(\\n                    split,\\n                    model_name=model_name,\\n                    epochs=args.epochs,\\n                    batch_size=args.batch_size,\\n                    max_length=args.max_length,\\n                )\\n            )\\n\\n    tag = \\\"smoke\\\" if args.smoke else \\\"full\\\"\\n    write_tables(rows, tag)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"eda.py\": \"\\\"\\\"\\\"Dataset EDA for thesis: sizes, balance, length, examples.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom collections import Counter\\nfrom pathlib import Path\\n\\nfrom src.data_load import load_task\\nfrom src.paths import ALL_TASKS, OUTPUTS\\n\\n\\nTASK_DESC = {\\n    \\\"seconvo\\\": \\\"SE chat (LinkedIn-style). Label: attack vs benign.\\\",\\n    \\\"mentalmanip\\\": \\\"Dialogue mental manipulation. Label: manipulative vs not.\\\",\\n    \\\"safepersuasion\\\": \\\"Online comments. Label: Manipulation vs Rational Persuasion.\\\",\\n    \\\"scam_phone\\\": \\\"Synthetic phone dialogues. Label: scam vs not (control / easy).\\\",\\n    \\\"phishing_text\\\": \\\"Email/SMS-like texts. Label: phishing vs benign.\\\",\\n}\\n\\n\\ndef _len_stats(texts: list[str]) -> dict:\\n    words = [len(t.split()) for t in texts]\\n    chars = [len(t) for t in texts]\\n    words_sorted = sorted(words)\\n    n = len(words_sorted)\\n\\n    def pct(p: float) -> int:\\n        return words_sorted[min(n - 1, int(n * p))]\\n\\n    return {\\n        \\\"n\\\": n,\\n        \\\"words_p50\\\": pct(0.5),\\n        \\\"words_p90\\\": pct(0.9),\\n        \\\"words_max\\\": max(words),\\n        \\\"chars_p50\\\": sorted(chars)[n // 2],\\n        \\\"frac_gt_256_words\\\": round(sum(1 for w in words if w > 256) / n, 4),\\n        \\\"frac_gt_512_words\\\": round(sum(1 for w in words if w > 512) / n, 4),\\n    }\\n\\n\\ndef analyze_task(task: str) -> dict:\\n    split = load_task(task)\\n    y_all = split.y_train + split.y_test\\n    bal = Counter(y_all)\\n    texts = split.texts_train + split.texts_test\\n    example_pos = next(\\n        (t for t, y in zip(texts, y_all) if y == 1),\\n        \\\"\\\",\\n    )\\n    example_neg = next(\\n        (t for t, y in zip(texts, y_all) if y == 0),\\n        \\\"\\\",\\n    )\\n    return {\\n        \\\"task\\\": task,\\n        \\\"description\\\": TASK_DESC.get(task, \\\"\\\"),\\n        \\\"n_train\\\": len(split.y_train),\\n        \\\"n_test\\\": len(split.y_test),\\n        \\\"n_total\\\": len(y_all),\\n        \\\"class_0\\\": int(bal.get(0, 0)),\\n        \\\"class_1\\\": int(bal.get(1, 0)),\\n        \\\"pos_rate\\\": round(bal.get(1, 0) / len(y_all), 4),\\n        \\\"length\\\": _len_stats(texts),\\n        \\\"example_pos\\\": \\\" \\\".join(example_pos.split())[:350],\\n        \\\"example_neg\\\": \\\" \\\".join(example_neg.split())[:350],\\n    }\\n\\n\\ndef run_eda(tasks: list[str] | None = None) -> list[dict]:\\n    tasks = tasks or list(ALL_TASKS)\\n    rows = [analyze_task(t) for t in tasks]\\n    OUTPUTS.mkdir(parents=True, exist_ok=True)\\n    out_json = OUTPUTS / \\\"eda_summary.json\\\"\\n    out_md = OUTPUTS / \\\"eda_summary.md\\\"\\n    out_json.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding=\\\"utf-8\\\")\\n\\n    lines = [\\n        \\\"# Dataset EDA\\\",\\n        \\\"\\\",\\n        \\\"| task | n_total | train/test | class_0 | class_1 | pos_rate | words_p50 | words_p90 | frac>512w |\\\",\\n        \\\"|---|---:|---:|---:|---:|---:|---:|---:|---:|\\\",\\n    ]\\n    for r in rows:\\n        L = r[\\\"length\\\"]\\n        lines.append(\\n            f\\\"| {r['task']} | {r['n_total']} | {r['n_train']}/{r['n_test']} | \\\"\\n            f\\\"{r['class_0']} | {r['class_1']} | {r['pos_rate']:.3f} | \\\"\\n            f\\\"{L['words_p50']} | {L['words_p90']} | {L['frac_gt_512_words']:.3f} |\\\"\\n        )\\n    lines.append(\\\"\\\")\\n    for r in rows:\\n        lines.extend(\\n            [\\n                f\\\"## {r['task']}\\\",\\n                \\\"\\\",\\n                r[\\\"description\\\"],\\n                \\\"\\\",\\n                f\\\"**pos:** {r['example_pos']}\\\",\\n                \\\"\\\",\\n                f\\\"**neg:** {r['example_neg']}\\\",\\n                \\\"\\\",\\n            ]\\n        )\\n    out_md.write_text(\\\"\\\\n\\\".join(lines), encoding=\\\"utf-8\\\")\\n    print(out_md.read_text(encoding=\\\"utf-8\\\"))\\n    print(f\\\"wrote {out_json}\\\")\\n    print(f\\\"wrote {out_md}\\\")\\n    return rows\\n\\n\\ndef main() -> None:\\n    run_eda()\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"compare_results.py\": \"\\\"\\\"\\\"Merge result JSON files into comparison pivot tables.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom collections import defaultdict\\nfrom pathlib import Path\\n\\nfrom src.paths import OUTPUTS\\n\\n\\ndef _load_rows(paths: list[Path]) -> list[dict]:\\n    rows: list[dict] = []\\n    for path in paths:\\n        data = json.loads(path.read_text(encoding=\\\"utf-8\\\"))\\n        if isinstance(data, list):\\n            rows.extend(data)\\n    return rows\\n\\n\\ndef pivot_f1(rows: list[dict], metric: str = \\\"f1_macro\\\") -> tuple[list[str], list[str], dict]:\\n    tasks = sorted({r[\\\"task\\\"] for r in rows})\\n    models = []\\n    seen = set()\\n    for r in rows:\\n        if r[\\\"model\\\"] not in seen:\\n            models.append(r[\\\"model\\\"])\\n            seen.add(r[\\\"model\\\"])\\n    grid: dict[tuple[str, str], float] = {}\\n    for r in rows:\\n        grid[(r[\\\"task\\\"], r[\\\"model\\\"])] = float(r[\\\"metrics\\\"][metric])\\n    return tasks, models, grid\\n\\n\\ndef to_markdown(tasks: list[str], models: list[str], grid: dict, metric: str) -> str:\\n    header = \\\"| task | \\\" + \\\" | \\\".join(models) + \\\" |\\\"\\n    sep = \\\"|---|\\\" + \\\"|\\\".join([\\\"---:\\\" for _ in models]) + \\\"|\\\"\\n    lines = [f\\\"# Comparison (`{metric}`)\\\", \\\"\\\", header, sep]\\n    for task in tasks:\\n        cells = []\\n        for model in models:\\n            val = grid.get((task, model))\\n            cells.append(f\\\"{val:.4f}\\\" if val is not None else \\\"\u2014\\\")\\n        lines.append(f\\\"| {task} | \\\" + \\\" | \\\".join(cells) + \\\" |\\\")\\n    return \\\"\\\\n\\\".join(lines) + \\\"\\\\n\\\"\\n\\n\\ndef main() -> None:\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\n        \\\"--glob\\\",\\n        default=\\\"results_*.json\\\",\\n        help=\\\"glob under outputs/\\\",\\n    )\\n    parser.add_argument(\\\"--metric\\\", default=\\\"f1_macro\\\", choices=[\\\"f1\\\", \\\"f1_macro\\\", \\\"accuracy\\\"])\\n    args = parser.parse_args()\\n\\n    paths = sorted(OUTPUTS.glob(args.glob))\\n    # Prefer full runs, skip smoke if both exist\\n    full = [p for p in paths if \\\"smoke\\\" not in p.name]\\n    paths = full or paths\\n    if not paths:\\n        raise SystemExit(f\\\"no files matched outputs/{args.glob}\\\")\\n\\n    rows = _load_rows(paths)\\n    # keep latest row per (task, model)\\n    latest: dict[tuple[str, str], dict] = {}\\n    for r in rows:\\n        latest[(r[\\\"task\\\"], r[\\\"model\\\"])] = r\\n    rows = list(latest.values())\\n\\n    tasks, models, grid = pivot_f1(rows, args.metric)\\n    md = to_markdown(tasks, models, grid, args.metric)\\n    out = OUTPUTS / f\\\"comparison_{args.metric}.md\\\"\\n    out.write_text(md, encoding=\\\"utf-8\\\")\\n    print(md)\\n    print(f\\\"wrote {out}\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\", \"__init__.py\": \"\\\"\\\"\\\"Thesis experiment: SEConvo + MentalManip, SVM vs BERT.\\\"\\\"\\\"\\n\"}")
for name, text in FILES.items():
    (SRC / name).write_text(text, encoding="utf-8")
    print("wrote src/" + name)

def fetch(url, dest, attempts=8):
    if dest.exists() and dest.stat().st_size > 1000:
        print("skip", dest.name, dest.stat().st_size)
        return
    print("download", dest.name)
    if "ealvaradob/phishing-dataset" in url and dest.name == "texts.json":
        try:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download(
                repo_id="ealvaradob/phishing-dataset",
                filename="texts.json",
                repo_type="dataset",
            )
            shutil.copy(path, dest)
            print(" ", dest.stat().st_size, "(hf_hub)")
            return
        except Exception as e:
            print(" hf_hub failed:", e)
    tmp = dest.with_suffix(dest.suffix + ".part")
    last = None
    for i in range(attempts):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "thesis-colab/2.0"})
            with urllib.request.urlopen(req, timeout=180) as resp, open(tmp, "wb") as out:
                shutil.copyfileobj(resp, out)
            shutil.move(tmp, dest)
            print(" ", dest.stat().st_size)
            return
        except urllib.error.HTTPError as e:
            last = e
            if tmp.exists():
                tmp.unlink()
            wait = 20 * (i + 1)
            if e.code in (429, 503):
                print(f"  HTTP {e.code}, wait {wait}s")
                time.sleep(wait)
                continue
            raise
        except Exception as e:
            last = e
            if tmp.exists():
                tmp.unlink()
            wait = 15 * (i + 1)
            print(f"  error {e}, wait {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"failed {dest.name}: {last}")

downloads = [
    ("https://zenodo.org/records/12170260/files/annotated_train.json?download=1", ROOT / "data/seconvo/annotated_train.json"),
    ("https://zenodo.org/records/12170260/files/annotated_test.json?download=1", ROOT / "data/seconvo/annotated_test.json"),
    ("https://huggingface.co/datasets/audreyeleven/MentalManip/resolve/main/mentalmanip_maj.csv?download=true", ROOT / "data/mentalmanip/mentalmanip_maj.csv"),
    ("https://raw.githubusercontent.com/haeinkong/SafePersuasion/main/dataset/SafePersuasion.csv", ROOT / "data/safepersuasion/SafePersuasion.csv"),
    ("https://huggingface.co/datasets/BothBosu/multi-agent-scam-conversation/resolve/main/agent_conversation_train.csv?download=true", ROOT / "data/scam_phone/agent_conversation_train.csv"),
    ("https://huggingface.co/datasets/BothBosu/multi-agent-scam-conversation/resolve/main/agent_conversation_test.csv?download=true", ROOT / "data/scam_phone/agent_conversation_test.csv"),
    ("https://huggingface.co/datasets/ealvaradob/phishing-dataset/resolve/main/texts.json?download=true", ROOT / "data/phishing/texts.json"),
]
for url, dest in downloads:
    fetch(url, dest)

import os, sys
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("cwd", os.getcwd())


## 2) EDA korpusów
Rozmiary, balans klas, długości tekstów, przykłady pos/neg.
Zwróć uwagę: **SEConvo i scam_phone** mają długie rozmowy → dlatego używamy `max_length=512`.


In [ ]:
import os, sys
os.chdir("/content/se-transformer")
sys.path.insert(0, "/content/se-transformer")
from src.eda import run_eda
eda_rows = run_eda()


## 3) Baseline: majority + TF-IDF + LinearSVC
Szybkie. Wyniki idą do `outputs/`.


In [ ]:
import os, sys
os.chdir("/content/se-transformer")
sys.path.insert(0, "/content/se-transformer")
!python -m src.run_all --task all --skip-transformers --max-length 512


## 4) Transformery (3 modele) — główna część badawcza
Kolejność: BERT → RoBERTa → DistilBERT, wszystkie zadania, **max_length=512**, 3 epoki.

Jeśli sesja padnie w połowie, odpal poniższą komórkę z `--models` tylko dla brakującego modelu, np.:
`!python -m src.run_all --task all --models roberta-base --epochs 3 --batch-size 8 --max-length 512`


In [ ]:
import os, sys
os.chdir("/content/se-transformer")
sys.path.insert(0, "/content/se-transformer")
# Pełny przebieg: majority+SVM jest już policzone wyżej; tu doklejamy 3 Transformery.
# Aby nie dublować SVM, można odpalić tylko modele — ale run_all zawsze liczy majority+SVM (szybkie).
!python -m src.run_all --task all --models bert-base-uncased,roberta-base,distilbert-base-uncased --epochs 3 --batch-size 8 --max-length 512


## 5) Tabele porównawcze (F1 macro i F1)
Pivot po wszystkich plikach `results_*.json` w `outputs/`.


In [ ]:
import os, sys
os.chdir("/content/se-transformer")
sys.path.insert(0, "/content/se-transformer")
!python -m src.compare_results --metric f1_macro
!python -m src.compare_results --metric f1
from pathlib import Path
print(Path("outputs/comparison_f1_macro.md").read_text(encoding="utf-8"))
print(Path("outputs/comparison_f1.md").read_text(encoding="utf-8"))


## 6) Jak czytać wyniki (do rozdziału Dyskusja)

| Zbiór | Rola w pracy |
|---|---|
| SafePersuasion | hard case manipulacji semantycznej — tu Transformer powinien pomagać |
| MentalManip | manipulacja w dialogu; raportuj **F1 macro** (imbalance) |
| SEConvo | SE w czacie; mały N; długie rozmowy → 512 tokenów |
| phishing_text | skala / klasyczny kanał SE |
| scam_phone | **kontrola** — jeśli F1≈1.0, nie jest główny dowód |

**Pytanie badawcze:** czy modele Transformer przewyższają baseline SVM w detekcji SE i manipulacji semantycznej — i kiedy nie.


## 7) Pobierz artefakty do magistry

In [ ]:
from pathlib import Path
from google.colab import files

outs = Path("/content/se-transformer/outputs")
for name in [
    "results_latest.md",
    "eda_summary.md",
    "eda_summary.json",
    "comparison_f1_macro.md",
    "comparison_f1.md",
]:
    p = outs / name
    if p.exists():
        print("download", p)
        files.download(str(p))

# najnowszy pełny JSON
jsons = sorted(outs.glob("results_full_*.json"))
if jsons:
    files.download(str(jsons[-1]))
print("DONE — część badawcza do wklejenia w rozdział Wyniki")
